# Peak shaving vs TCIPC analysis

Let's check the results. First we need to copy the results from the cluster to
this PC:
```
rsync -avm --include='*/' --include='*.sql*' --exclude='*' jhummel@login.delftblue.tudelft.nl:../../scratch/jhummel/tip_clearance/data/optimal_tuning/ ./data/optimal_tuning/ --dry-run
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.interpolate import griddata
from weis.visualization.utils import load_OMsql

plt.style.use("journal.mplstyle")

%matplotlib widget

In [ ]:
# Define the logs to load and give them labels.
logs_to_load = {
    "Baseline": "../../../data/optimal_tuning/parameter_sweep/ps_vs_ipc_baseline.sql",
    "Free yaw": "../../../data/optimal_tuning/parameter_sweep/ps_vs_ipc_free_yaw.sql",
    "Zero yaw": "../../../data/optimal_tuning/parameter_sweep/ps_vs_ipc_zero_yaw.sql",
}

# Load all datasets
all_data_dicts = {}
for log_name, log_fmt in logs_to_load.items():
    all_data_dicts[log_name] = load_OMsql(log_fmt)
    print(f"Loaded {log_name}: {all_data_dicts[log_name].keys()}")

In [ ]:
# Let's define how we load, scale, and label the data, then make a dataframe.
all_outputs = {
    # ROSCO variables.
    "TCIPC_MaxTipDeflection": {
        "key": "tune_rosco_ivc.TCIPC_MaxTipDeflection",
        "scaling": lambda x: x[0],
        "label": "TCIPC max tip deflection (m)",
    },
    "ps_percent": {
        "key": "tune_rosco_ivc.ps_percent",
        "scaling": lambda x: x[0],
        "label": "Peak shaving (-)",
    },
    "TCIPC_nHarmonics": {
        "key": "tune_rosco_ivc.TCIPC_nHarmonics",
        "scaling": lambda x: x[0],
        "label": "Number of harmonics",
    },
    "TCIPC_ZeroYawDeflection": {
        "key": "tune_rosco_ivc.TCIPC_ZeroYawDeflection",
        "scaling": lambda x: x[0],
        "label": "Zero yaw deflection",
    },
    # Objectives / responses.
    "aep": {
        "key": "aeroelastic.AEP",
        "scaling": lambda x: 1e-6 * x[0],
        "label": "AEP (GWh)",
    },
    "max_TipDxc_towerPassing": {
        "key": "aeroelastic.max_eff_TipDxc_towerPassing",
        "scaling": lambda x: x[0],
        "label": "Max TipDxc tower passing (m)",
    },
    "tower_clearance": {
        "key": "aeroelastic.max_eff_TipDxc_towerPassing",
        "scaling": lambda x: 30 - x[0],
        "label": "Tower clearance (m)",
    },
    "TCIPC_amplitude_at_max_deflection": {
        "key": "aeroelastic.TCIPC_amplitude_at_max_deflection",
        "scaling": lambda x: x[0],
        "label": "TCIPC amplitude (deg)",
    },
    # Structural loads.
    "max_TwrBsMyt": {
        "key": "aeroelastic.max_TwrBsMyt",
        "scaling": lambda x: x[0] / 1000,
        "label": "Max tower base Myt (MNm)",
    },
    "DEL_TwrBsMyt": {
        "key": "aeroelastic.DEL_TwrBsMyt",
        "scaling": lambda x: x[0] / 1000,
        "label": "DEL tower base Myt (MNm)",
    },
    "damage_tower_base": {
        "key": "aeroelastic.damage_tower_base",
        "scaling": lambda x: x[0],
        "label": "Tower base damage (-)",
    },
    "max_RootMyb": {
        "key": "aeroelastic.max_RootMyb",
        "scaling": lambda x: x[0] / 1000,
        "label": "Max root Myb (MNm)",
    },
    "DEL_RootMyb": {
        "key": "aeroelastic.DEL_RootMyb",
        "scaling": lambda x: x[0] / 1000,
        "label": "DEL root Myb (MNm)",
    },
    "hub_Mxyz": {
        "key": "aeroelastic.hub_Mxyz",
        "scaling": lambda x: np.linalg.norm(x) / 1000,
        "label": "Hub moment magnitude (MNm)",
    },
    # Control activity.
    "max_pitch_rate_sim": {
        "key": "aeroelastic.max_pitch_rate_sim",
        "scaling": lambda x: x[0],
        "label": "Max pitch rate (deg/s)",
    },
    "avg_pitch_travel": {
        "key": "aeroelastic.avg_pitch_travel",
        "scaling": lambda x: x[0],
        "label": "Avg pitch travel (deg)",
    },
}

# Build dataframe from mapping for each log.
labels = {short: info["label"] for short, info in all_outputs.items()}
all_dfs = []

for log_name, data_dict in all_data_dicts.items():
    df_dict = {}
    for short_label, info in all_outputs.items():
        data = data_dict[info["key"]]
        scaled_data = list(map(info["scaling"], data))
        df_dict[short_label] = scaled_data

    df_temp = pd.DataFrame(df_dict)
    df_temp["log_name"] = log_name
    all_dfs.append(df_temp)

# Combine all dataframes.
df = pd.concat(all_dfs, ignore_index=True)
print(f"Combined dataframe shape: {df.shape}")

# Constants used throughout the notebook.
BASELINE_LOG = "Baseline"
REFERENCE_POINT = (0.8, 0.0)
DESIGN_VARS = ("ps_percent", "TCIPC_MaxTipDeflection")

# The baseline only varies ps_percent (no TCIPC control), so its rows share a
# single TCIPC_MaxTipDeflection value. Replicate them across the TCIPC grid
# from the 2D datasets to produce a flat response surface, which is physically
# correct because the baseline controller does not use TCIPC.
_bl_mask = df["log_name"] == BASELINE_LOG
_tip_values = sorted(df[~_bl_mask]["TCIPC_MaxTipDeflection"].unique())
_df_bl = df[_bl_mask].copy()
_replicated = [_df_bl.assign(TCIPC_MaxTipDeflection=t) for t in _tip_values]
df = pd.concat([df[~_bl_mask]] + _replicated, ignore_index=True)
print(f"After replicating baseline across TCIPC grid: {df.shape}")
df.head()

## Data exploration

In [ ]:
# Make a plot of the distribution of our design variables.
plt.figure()
sns.scatterplot(
    df,
    x="TCIPC_MaxTipDeflection",
    y="ps_percent",
    style="log_name",
)
plt.show()

In [ ]:
# Plot several outputs/objectives as a function of the design variables.
outputs = [
    "aep",
    "tower_clearance",
    "TCIPC_amplitude_at_max_deflection",
    "max_TwrBsMyt",
    "DEL_TwrBsMyt",
    "damage_tower_base",
    "max_RootMyb",
    "DEL_RootMyb",
    "hub_Mxyz",
    "max_pitch_rate_sim",
    "avg_pitch_travel",
]

fig, axs = plt.subplots(len(outputs), len(logs_to_load), figsize=(10, 30))

for i, output in enumerate(outputs):
    for j, log_name in enumerate(logs_to_load.keys()):
        scatter = axs[i, j].scatter(
            df[df["log_name"] == log_name]["ps_percent"],
            df[df["log_name"] == log_name]["TCIPC_MaxTipDeflection"],
            c=df[df["log_name"] == log_name][output],
        )

        axs[i, j].set_xlabel(labels["ps_percent"])
        axs[i, j].set_ylabel(labels["TCIPC_MaxTipDeflection"])
        plt.colorbar(scatter, ax=axs[i, j])

        if i == 0:
            axs[i, j].set_title(log_name)
        if j == 0:
            axs[i, j].annotate(
                labels[output],
                xy=(0, 0.5),
                xytext=(-axs[i, j].yaxis.labelpad - 5, 0),
                xycoords=axs[i, j].yaxis.label,
                textcoords="offset points",
                ha="right",
                va="center",
                rotation=90,
            )

plt.show()

In [ ]:
# Get an idea of the trade-off between AEP and tower clearance.
fig, ax = plt.subplots()

df_sorted = df.sort_values("aep")

sns.lineplot(
    data=df_sorted[df_sorted["TCIPC_MaxTipDeflection"] == 0.0],
    x="aep",
    y="tower_clearance",
    hue="log_name",
)
sns.lineplot(
    data=df_sorted[df_sorted["TCIPC_MaxTipDeflection"] == 20.0],
    x="aep",
    y="tower_clearance",
    hue="log_name",
    palette="deep",
)

## Data interpolation

In [ ]:
# Define the bounds of our design space for interpolation.
ps_min, ps_max = (
    df["ps_percent"].min(),
    df["ps_percent"].max(),
)
tip_min, tip_max = (
    df["TCIPC_MaxTipDeflection"].min(),
    df["TCIPC_MaxTipDeflection"].max(),
)

# Create a regular grid for interpolation to enable smooth contour plots.
n_points = 50
ps_grid = np.linspace(ps_min, ps_max, n_points)
tip_grid = np.linspace(tip_min, tip_max, n_points)
ps_percent_grid, tcipc_reference_grid = np.meshgrid(ps_grid, tip_grid)

# Interpolate each output variable on the grid for each log.
interpolated_data = {}

for log_name in logs_to_load.keys():
    interpolated_data[log_name] = {}
    df_log = df[df["log_name"] == log_name]

    points = df_log[["ps_percent", "TCIPC_MaxTipDeflection"]].values

    for output in outputs:
        values = df_log[output].values

        grid_values = griddata(
            points,
            values,
            (ps_percent_grid, tcipc_reference_grid),
            method="cubic",
        )

        interpolated_data[log_name][output] = grid_values

print(f"Interpolated {len(outputs)} outputs for {len(logs_to_load)} datasets")
print(f"Grid shape: {ps_percent_grid.shape}")

In [ ]:
# Plot interpolated contour surfaces for each output and dataset.
fig, axs = plt.subplots(len(outputs), len(logs_to_load), figsize=(10, 30))

for i, output in enumerate(outputs):
    # Compute shared color limits across both datasets.
    all_vals = np.concatenate(
        [interpolated_data[ln][output].ravel() for ln in logs_to_load]
    )
    vmin = np.nanmin(all_vals)
    vmax = np.nanmax(all_vals)
    levels = np.linspace(vmin, vmax, 9)

    # Compute the reference value for this output from the baseline dataset.
    df_bl = df[df["log_name"] == BASELINE_LOG]
    ref_val = griddata(
        df_bl[["ps_percent", "TCIPC_MaxTipDeflection"]].values,
        df_bl[output].values,
        [REFERENCE_POINT],
        method="linear",
    )[0]

    for j, log_name in enumerate(logs_to_load.keys()):
        contour = axs[i, j].contourf(
            ps_percent_grid,
            tcipc_reference_grid,
            interpolated_data[log_name][output],
            levels=levels,
            vmin=vmin,
            vmax=vmax,
        )

        # Iso line at the reference value.
        axs[i, j].contour(
            ps_percent_grid,
            tcipc_reference_grid,
            interpolated_data[log_name][output],
            levels=[ref_val],
            colors="k",
            linewidths=0.8,
        )

        axs[i, j].set_xlabel(labels["ps_percent"])
        axs[i, j].set_ylabel(labels["TCIPC_MaxTipDeflection"])
        plt.colorbar(contour, ax=axs[i, j])

        if i == 0:
            axs[i, j].set_title(log_name)
        if j == 0:
            axs[i, j].annotate(
                labels[output],
                xy=(0, 0.5),
                xytext=(-axs[i, j].yaxis.labelpad - 5, 0),
                xycoords=axs[i, j].yaxis.label,
                textcoords="offset points",
                ha="right",
                va="center",
                rotation=90,
            )

In [ ]:
# Investigate one plot in detail.
fig, ax = plt.subplots()

scatter = ax.contourf(
    ps_percent_grid,
    tcipc_reference_grid,
    interpolated_data["Free yaw"]["DEL_RootMyb"],
    levels=np.linspace(18, 25, 100),
    # levels=np.linspace(0.15, 1.5, 100),
    # levels=np.linspace(260, 420, 100),
)

# plt.title("DEL_RootMyb")
plt.colorbar(scatter)

## Optimization

In [ ]:
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.unsga3 import UNSGA3
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.algorithms.moo.ctaea import CTAEA
from pymoo.algorithms.moo.sms import SMSEMOA
from pymoo.algorithms.moo.age2 import AGEMOEA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.termination import get_termination
from pymoo.optimize import minimize
from pymoo.indicators.hv import Hypervolume
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator
from copy import deepcopy
from itertools import cycle

In [ ]:
class InterpolatorSet:
    """Builds interpolators for all numeric columns of a dataframe subset.

    Given a dataframe (typically filtered to one log_name), this creates a
    scipy interpolator for each numeric column, using two specified design
    variable columns as inputs.
    """

    def __init__(self, df, design_vars, method="linear"):
        xy = df[list(design_vars)].values

        Interpolator = (
            CloughTocher2DInterpolator if method == "cubic" else LinearNDInterpolator
        )

        # Build an interpolator for every numeric column that is not a
        # design variable.
        self._interpolators = {}
        for col in df.select_dtypes(include=[np.number]).columns:
            if col in design_vars:
                continue
            self._interpolators[col] = Interpolator(xy, df[col].values)

    def __getitem__(self, column):
        return self._interpolators[column]

In [ ]:
class OptimizationProblem:
    """Declarative specification of a multi-objective optimization problem.

    Objectives are specified as dicts:
        {"variable": str, "direction": "minimize" | "maximize", "label": str}

    Constraints are specified as dicts:
        {
            "name": str,           # Descriptive name for plotting.
            "variable": str,       # Column name from InterpolatorSet.
            "label": str,          # Axis label including units.
            "threshold": float,    # Pre-computed absolute threshold value.
            "direction": "below" | "above"  # "below" = must stay below threshold.
        }
    """

    def __init__(self, objectives, constraints, xl, xu):
        self.objectives = objectives
        self.constraints = constraints
        self.xl = np.array(xl)
        self.xu = np.array(xu)

    def to_pymoo(self, interp_set):
        """Build a pymoo ElementwiseProblem from this specification.

        Pymoo always minimizes, so objectives with direction="maximize" are
        negated internally.
        """
        objectives = self.objectives
        constraints = self.constraints
        xl = self.xl
        xu = self.xu

        # Build objective callables. Negate "maximize" objectives so pymoo
        # can minimize them.
        obj_callables = []
        for obj in objectives:
            interp = interp_set[obj["variable"]]
            sign = -1.0 if obj["direction"] == "maximize" else 1.0
            obj_callables.append(lambda x, y, _i=interp, _s=sign: _s * _i(x, y))

        # Build constraint callables. G <= 0 is feasible.
        con_callables = []
        for con in constraints:
            interp = interp_set[con["variable"]]
            threshold = con["threshold"]
            if con["direction"] == "below":
                # Feasible when current <= threshold.
                con_callables.append(
                    lambda x, y, _i=interp, _t=threshold: _i(x, y) - _t
                )
            else:
                # Feasible when current >= threshold.
                con_callables.append(
                    lambda x, y, _i=interp, _t=threshold: _t - _i(x, y)
                )

        class _Problem(ElementwiseProblem):
            def __init__(self):
                super().__init__(
                    n_var=2,
                    n_obj=len(objectives),
                    n_ieq_constr=len(constraints),
                    xl=xl,
                    xu=xu,
                )

            def _evaluate(self, x, out, *args, **kwargs):
                out["F"] = [f(*x) for f in obj_callables]
                out["G"] = [g(*x) for g in con_callables]

        return _Problem()

In [ ]:
class OptimizationStudy:
    """Runs multi-objective optimization across datasets and algorithms.

    Runs the OptimizationProblem on every log_name in the dataframe, using
    every algorithm in the provided dict. For the baseline dataset, the second
    design variable (TCIPC) is fixed at zero since it does not use TCIPC.
    """

    MARKERS = ["o", "s", "^", "D", "v", "P", "*", "X"]

    def __init__(
        self,
        df,
        problem,
        algorithms,
        termination,
        design_vars,
        baseline_log,
        interp_method="linear",
        seed=1,
        verbose=False,
    ):
        self.df = df
        self.problem = problem
        self.algorithms = algorithms
        self.termination = termination
        self.design_vars = design_vars
        self.baseline_log = baseline_log
        self.interp_method = interp_method
        self.seed = seed
        self.verbose = verbose

        self.log_names = sorted(df["log_name"].unique().tolist())
        self.results = {}
        self.interp_sets = {}

    def run(self):
        """Run all optimizations for each (log_name, algorithm) pair."""
        # Build interpolator sets for all datasets.
        for log_name in self.log_names:
            df_log = self.df[self.df["log_name"] == log_name]
            self.interp_sets[log_name] = InterpolatorSet(
                df_log, self.design_vars, method=self.interp_method
            )

        # Run optimizations for each dataset and algorithm.
        for log_name in self.log_names:
            # The baseline does not use TCIPC, so fix it at zero.
            if log_name == self.baseline_log:
                prob = deepcopy(self.problem)
                prob.xl[1] = 0.0
                prob.xu[1] = 0.0
            else:
                prob = self.problem

            interp_set = self.interp_sets[log_name]
            for algo_name, algo_factory in self.algorithms.items():
                result = minimize(
                    prob.to_pymoo(interp_set),
                    deepcopy(algo_factory),
                    deepcopy(self.termination),
                    seed=self.seed,
                    save_history=True,
                    verbose=self.verbose,
                )
                self.results[(log_name, algo_name)] = result
                if result.F is None:
                    print("Didn't get any results...")
                else:
                    print(f"  Done: {log_name} -> {len(result.F)} solutions")

    def _calculate_hv_history(self, result, ref_point=None):
        """Compute hypervolume at each generation from a pymoo result.

        The ref_point should be given in the original objective space (before
        the sign flip that pymoo uses internally for maximization). It is
        converted to the minimization space automatically.
        """
        hist_F = []
        for algo in result.history:
            opt = algo.opt
            feas = np.where(opt.get("feasible"))[0]
            hist_F.append(opt.get("F")[feas])

        if ref_point is not None:
            # Convert from original objective space to pymoo's minimization
            # space by applying the same sign flip used for maximized objectives.
            signs = np.array(
                [
                    -1.0 if o["direction"] == "maximize" else 1.0
                    for o in self.problem.objectives
                ]
            )
            pymoo_ref = np.array(ref_point) * signs
        else:
            pymoo_ref = result.F.max(axis=0)

        metric = Hypervolume(
            ref_point=pymoo_ref,
            norm_ref_point=False,
            zero_to_one=False,
        )
        return [metric.do(_F) for _F in hist_F]

    def plot_convergence(self, ref_point=None, normalize=False, ax=None, legend=True):
        """Plot hypervolume convergence for all optimization runs.

        ref_point sets the hypervolume reference point in the original
        objective space. When provided, the absolute hypervolume values
        are comparable across scenarios.

        If normalize is True, each run's hypervolume is scaled to [0, 1].
        """
        if ax is None:
            fig, ax = plt.subplots()
        else:
            fig = ax.get_figure()
        markers = cycle(self.MARKERS)

        for (log_name, algo_name), result in self.results.items():
            hv = np.array(self._calculate_hv_history(result, ref_point=ref_point))

            if normalize:
                hv_min, hv_max = hv[0], hv[-1]
                if hv_max > hv_min:
                    hv = (hv - hv_min) / (hv_max - hv_min)

            ax.plot(
                hv,
                marker=next(markers),
                markersize=3,
                label=f"{log_name}",
            )

        ax.set_xlabel("Generation")
        ax.set_ylabel("Normalized hypervolume (-)" if normalize else "Hypervolume")
        if legend:
            ax.legend()
        return fig, ax

    def plot_pareto(self, ax=None, legend=True):
        """Plot Pareto front comparison for all runs (2 objectives).

        Objective values are converted back to the original direction by
        un-negating maximized objectives.
        """
        signs = np.array(
            [
                -1.0 if o["direction"] == "maximize" else 1.0
                for o in self.problem.objectives
            ]
        )

        if ax is None:
            fig, ax = plt.subplots()
        else:
            fig = ax.get_figure()
        markers = cycle(self.MARKERS)

        for (log_name, algo_name), result in self.results.items():
            F = result.F * signs
            sort_idx = F[:, 0].argsort()
            F = F[sort_idx]
            ax.plot(
                F[:, 0],
                F[:, 1],
                marker=next(markers),
                markersize=4,
                label=f"{log_name}",
            )

        ax.set_xlabel(self.problem.objectives[0]["label"])
        ax.set_ylabel(self.problem.objectives[1]["label"])
        if legend:
            ax.legend()
        return fig, ax

    def plot_design_space(self, ax=None, legend=True):
        """Plot design variable values for all Pareto-optimal points."""
        if ax is None:
            fig, ax = plt.subplots()
        else:
            fig = ax.get_figure()
        markers = cycle(self.MARKERS)

        for (log_name, algo_name), result in self.results.items():
            ax.scatter(
                result.X[:, 0],
                result.X[:, 1],
                marker=next(markers),
                s=20,
                label=f"{log_name}",
            )
        ax.set_xlim(self.problem.xl[0], self.problem.xu[0])
        ax.set_ylim(0, self.problem.xu[1])
        ax.set_xlabel(labels.get(self.design_vars[0], self.design_vars[0]))
        ax.set_ylabel(labels.get(self.design_vars[1], self.design_vars[1]))
        if legend:
            ax.legend()
        return fig, ax

    def plot_constraints(self, max_cols=3, sort=None):
        """Plot constraint satisfaction for Pareto-optimal solutions.

        Creates one subplot per constraint with at most max_cols per row.
        A shared legend is placed outside the top-right subplot.

        When sort is set to an objective variable name (e.g. "aep"), the
        x-axis is sorted by that objective instead of using the raw index.
        """
        # Pre-compute sort indices per result when requested.
        sort_indices = {}
        if sort:
            signs = np.array(
                [
                    -1.0 if o["direction"] == "maximize" else 1.0
                    for o in self.problem.objectives
                ]
            )
            # Find the column index for the requested objective.
            obj_names = [o["variable"] for o in self.problem.objectives]
            sort_col = obj_names.index(sort)
            for key, result in self.results.items():
                F = result.F * signs
                sort_indices[key] = F[:, sort_col].argsort()

        n_constraints = len(self.problem.constraints)
        ncols = min(n_constraints, max_cols)
        nrows = int(np.ceil(n_constraints / ncols))
        fig, axs = plt.subplots(nrows, ncols, squeeze=False, figsize=(7, 7))

        markers = cycle(self.MARKERS)
        # Pre-assign a marker per result so it is consistent across subplots.
        result_markers = {key: next(markers) for key in self.results}

        for j, con in enumerate(self.problem.constraints):
            row, col = divmod(j, ncols)
            ax = axs[row, col]
            threshold = con["threshold"]

            for (log_name, algo_name), result in self.results.items():
                key = (log_name, algo_name)
                # Recover the absolute value from the pymoo G output.
                # For "below": G = current - threshold, so current = G + threshold.
                # For "above": G = threshold - current, so current = threshold - G.
                if con["direction"] == "below":
                    values = result.G[:, j] + threshold
                else:
                    values = threshold - result.G[:, j]

                if sort and key in sort_indices:
                    values = values[sort_indices[key]]

                ax.scatter(
                    np.arange(len(values)),
                    values,
                    marker=result_markers[key],
                    s=15,
                    label=f"{log_name}",
                )

            ax.axhline(threshold, color="red", linestyle="--", label="Threshold")
            x_label = f"Solution index (sorted by {sort})" if sort else "Solution index"
            ax.set_xlabel(x_label)
            ax.set_ylabel(con["label"])
            ax.set_title(con["name"])

        # Hide unused axes when constraints don't fill the last row.
        for idx in range(n_constraints, nrows * ncols):
            row, col = divmod(idx, ncols)
            axs[row, col].set_visible(False)

        # Shared legend outside the top-right subplot.
        handles, leg_labels = axs[0, 0].get_legend_handles_labels()
        axs[0, ncols - 1].legend(
            handles,
            leg_labels,
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
        )
        return fig, axs

In [ ]:
# Compute constraint thresholds at the reference operating point.
# The reference settings are the standard peak-shaving-only controller.
reference_interp = InterpolatorSet(
    df[df["log_name"] == BASELINE_LOG], DESIGN_VARS, method="linear"
)

constraint_vars = [
    "max_TwrBsMyt",
    "DEL_TwrBsMyt",
    "damage_tower_base",
    "max_RootMyb",
    "DEL_RootMyb",
    "hub_Mxyz",
]

ref_values = {}
for var in constraint_vars:
    ref_values[var] = float(reference_interp[var](*REFERENCE_POINT))

print(f"Reference settings ({BASELINE_LOG} at {REFERENCE_POINT}):")
for var, val in ref_values.items():
    print(f"  {var:20s} = {val:.4f} {labels[var].split('(')[-1].rstrip(')')}")

In [ ]:
# Define the optimization problem.
# All structural loads are constrained to not exceed the reference settings.
# Pitch actuation signals (max_pitch_rate_sim, avg_pitch_travel) are unconstrained.
problem = OptimizationProblem(
    objectives=[
        {
            "variable": "aep",
            "direction": "maximize",
            "label": "AEP (GWh)",
        },
        {
            "variable": "tower_clearance",
            "direction": "maximize",
            "label": "Tower clearance (m)",
        },
    ],
    constraints=[
        {
            "name": "Max tower base Myt",
            "variable": "max_TwrBsMyt",
            "label": "Max tower base Myt (MNm)",
            "threshold": ref_values["max_TwrBsMyt"] * 1.0,
            "direction": "below",
        },
        {
            "name": "DEL tower base Myt",
            "variable": "DEL_TwrBsMyt",
            "label": "DEL tower base Myt (MNm)",
            "threshold": ref_values["DEL_TwrBsMyt"] * 1.0,
            "direction": "below",
        },
        # {
        #     "name": "Tower base damage",
        #     "variable": "damage_tower_base",
        #     "label": "Tower base damage (-)",
        #     "threshold": ref_values["damage_tower_base"] * 1.0,
        #     "direction": "below",
        # },
        {
            "name": "Max root Myb",
            "variable": "max_RootMyb",
            "label": "Max root Myb (MNm)",
            "threshold": ref_values["max_RootMyb"] * 1.0,
            "direction": "below",
        },
        {
            "name": "DEL root Myb",
            "variable": "DEL_RootMyb",
            "label": "DEL root Myb (MNm)",
            "threshold": ref_values["DEL_RootMyb"] * 1.0,
            "direction": "below",
        },
        # {
        #     "name": "Hub moment",
        #     "variable": "hub_Mxyz",
        #     "label": "Hub moment magnitude (MNm)",
        #     "threshold": ref_values["hub_Mxyz"] * 1.0,
        #     "direction": "below",
        # },
    ],
    xl=[0.5, 0.0],
    xu=[1.0, 20.0],
)

# Specify the algorithms to compare.
ref_dirs = get_reference_directions("uniform", 2, n_partitions=99)

algorithms = {
    "NSGA2": NSGA2(
        pop_size=100,
        # n_offsprings=50,
        # sampling=FloatRandomSampling(),
        # crossover=SBX(prob=0.9, eta=15),
        # mutation=PM(eta=15),
        # eliminate_duplicates=True,
    ),
    # "CTAEA": CTAEA(
    #     ref_dirs,
    #     sampling=FloatRandomSampling(),
    #     crossover=SBX(prob=0.9, eta=15),
    #     mutation=PM(eta=15),
    #     eliminate_duplicates=True,
    # ),
    # "NSGA3": NSGA3(
    #     pop_size=100,
    #     ref_dirs=ref_dirs,
    # ),
    # "UNSGA3": UNSGA3(
    #     pop_size=100,
    #     ref_dirs=ref_dirs,
    # # ),
    # "SMSEMOA": SMSEMOA(pop_size=100),
    # "AGEMOEA2": AGEMOEA2(pop_size=100),
}

termination = get_termination("n_gen", 100)

# Run the study across all datasets and algorithms.
study = OptimizationStudy(
    df,
    problem=problem,
    algorithms=algorithms,
    termination=termination,
    design_vars=DESIGN_VARS,
    baseline_log=BASELINE_LOG,
    interp_method="cubic",
)
study.run()

In [ ]:
# Hypervolume convergence comparison across all runs.
study.plot_convergence(ref_point=(0, 0), normalize=False)
plt.show()

In [ ]:
# Pareto front comparison across all runs.
study.plot_pareto()
plt.show()

In [ ]:
# Design space comparison across all runs.
study.plot_design_space()
plt.show()

In [ ]:
# Constraint satisfaction visualization.
study.plot_constraints()
plt.show()

## Journal plots

In [ ]:
from pathlib import Path
from cmcrameri import cm

FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Retrieve default figure dimensions from the style sheet.
default_width, default_height = plt.rcParams["figure.figsize"]

In [ ]:
# 2x3 contour plot: AEP and tower clearance as rows, one column per dataset.
journal_outputs = ["aep", "tower_clearance"]
journal_logs = list(logs_to_load.keys())

fig, axs = plt.subplots(
    len(journal_outputs),
    len(journal_logs),
    figsize=(default_width, 1.5 * default_height),
)

for i, output in enumerate(journal_outputs):
    # Shared colour limits across all datasets for this output.
    all_vals = np.concatenate(
        [interpolated_data[ln][output].ravel() for ln in journal_logs]
    )
    vmin = np.nanmin(all_vals)
    vmax = np.nanmax(all_vals)
    levels = np.linspace(vmin, vmax, 9)

    # Reference value from the baseline dataset.
    df_bl = df[df["log_name"] == BASELINE_LOG]
    ref_val = griddata(
        df_bl[["ps_percent", "TCIPC_MaxTipDeflection"]].values,
        df_bl[output].values,
        [REFERENCE_POINT],
        method="linear",
    )[0]

    for j, log_name in enumerate(journal_logs):
        contour = axs[i, j].contourf(
            ps_percent_grid,
            tcipc_reference_grid,
            interpolated_data[log_name][output],
            levels=levels,
            vmin=vmin,
            vmax=vmax,
            cmap=cm.batlow,
        )

        # Iso line at the reference value.
        axs[i, j].contour(
            ps_percent_grid,
            tcipc_reference_grid,
            interpolated_data[log_name][output],
            levels=[ref_val],
            colors="k",
            linewidths=0.8,
        )

        axs[i, j].set_xlabel(labels["ps_percent"])
        axs[i, j].set_ylabel(labels["TCIPC_MaxTipDeflection"])
        plt.colorbar(contour, ax=axs[i, j], format="%.0f")

        if i == 0:
            axs[i, j].set_title(log_name)
        if j == 0:
            axs[i, j].annotate(
                labels[output],
                xy=(0, 0.5),
                xytext=(-axs[i, j].yaxis.labelpad - 5, 0),
                xycoords=axs[i, j].yaxis.label,
                textcoords="offset points",
                ha="right",
                va="center",
                rotation=90,
            )
            for ax in axs.flat:
                ax.set_yticks(np.arange(0, 21, 5))
fig.savefig(FIGURE_DIR / "contour_aep_clearance.pdf")
plt.show()

### Unconstrained optimization

In [ ]:
# Unconstrained problem: same objectives, no structural-load constraints.
problem_unconstrained = OptimizationProblem(
    objectives=[
        {
            "variable": "aep",
            "direction": "maximize",
            "label": "AEP (GWh)",
        },
        {
            "variable": "tower_clearance",
            "direction": "maximize",
            "label": "Tower clearance (m)",
        },
    ],
    constraints=[],
    xl=[0.5, 0.0],
    xu=[1.0, 20.0],
)

study_unconstrained = OptimizationStudy(
    df,
    problem=problem_unconstrained,
    algorithms={"NSGA2": NSGA2(pop_size=100)},
    termination=get_termination("n_gen", 50),
    design_vars=DESIGN_VARS,
    baseline_log=BASELINE_LOG,
    interp_method="cubic",
)
study_unconstrained.run()

In [ ]:
fig, ax = plt.subplots(figsize=(default_width, 0.6 * default_height))
study_unconstrained.plot_convergence(ref_point=(0, 0), ax=ax, legend=False)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.savefig(FIGURE_DIR / "unconstrained_convergence.pdf")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2)
study_unconstrained.plot_pareto(ax=axs[0], legend=False)
study_unconstrained.plot_design_space(ax=axs[1])
fig.savefig(FIGURE_DIR / "unconstrained_pareto_design.pdf")
plt.show()

### Constrained optimization

In [ ]:
# Unconstrained problem: same objectives, no structural-load constraints.
problem_constrained = OptimizationProblem(
    objectives=[
        {
            "variable": "aep",
            "direction": "maximize",
            "label": "AEP (GWh)",
        },
        {
            "variable": "tower_clearance",
            "direction": "maximize",
            "label": "Tower clearance (m)",
        },
    ],
    constraints=[
        {
            "name": "Max tower base Myt",
            "variable": "max_TwrBsMyt",
            "label": "Max tower base Myt (MNm)",
            "threshold": ref_values["max_TwrBsMyt"] * 1.0,
            "direction": "below",
        },
        {
            "name": "DEL tower base Myt",
            "variable": "DEL_TwrBsMyt",
            "label": "DEL tower base Myt (MNm)",
            "threshold": ref_values["DEL_TwrBsMyt"] * 1.0,
            "direction": "below",
        },
        # {
        #     "name": "Tower base damage",
        #     "variable": "damage_tower_base",
        #     "label": "Tower base damage (-)",
        #     "threshold": ref_values["damage_tower_base"] * 1.0,
        #     "direction": "below",
        # },
        {
            "name": "Max root Myb",
            "variable": "max_RootMyb",
            "label": "Max root Myb (MNm)",
            "threshold": ref_values["max_RootMyb"] * 1.0,
            "direction": "below",
        },
        {
            "name": "DEL root Myb",
            "variable": "DEL_RootMyb",
            "label": "DEL root Myb (MNm)",
            "threshold": ref_values["DEL_RootMyb"] * 1.0,
            "direction": "below",
        },
        # {
        #     "name": "Hub moment",
        #     "variable": "hub_Mxyz",
        #     "label": "Hub moment magnitude (MNm)",
        #     "threshold": ref_values["hub_Mxyz"] * 1.0,
        #     "direction": "below",
        # },
    ],
    xl=[0.5, 0.0],
    xu=[1.0, 20.0],
)

study_constrained = OptimizationStudy(
    df,
    problem=problem_constrained,
    algorithms={"NSGA2": NSGA2(pop_size=100)},
    termination=get_termination("n_gen", 50),
    design_vars=DESIGN_VARS,
    baseline_log=BASELINE_LOG,
    interp_method="cubic",
)
study_constrained.run()

In [ ]:
fig, ax = plt.subplots(figsize=(default_width, 0.6 * default_height))
study_constrained.plot_convergence(ref_point=(0, 0), ax=ax, legend=False)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.savefig(FIGURE_DIR / "constrained_convergence.pdf")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2)
study_constrained.plot_pareto(ax=axs[0], legend=False)
study_constrained.plot_design_space(ax=axs[1])
fig.savefig(FIGURE_DIR / "constrained_pareto_design.pdf")
plt.show()

In [ ]:
fig, axs = study_constrained.plot_constraints(max_cols=2, sort="aep")
fig.savefig(FIGURE_DIR / "constrained_constraints.pdf")
plt.show()